# Flight duration model: Just distance

In this exercise you'll build a regression model to predict flight duration (the `duration` column).

For the moment you'll keep the model simple, including only the distance of the flight (the `km` column) as a predictor.

The data are in `flights`. The first few records are displayed in the terminal. These data have also been split into training and testing sets and are available as `flights_train` and `flights_test`.

## Instructions

- Create a linear regression object. Specify the name of the label column. Fit it to the training data.
- Make predictions on the testing data.
- Create a regression evaluator object and use it to evaluate RMSE on the testing data.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [5]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M3-Regression/2_Regression/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')

from pyspark.ml.feature import StringIndexer

flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

# from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import OneHotEncoderEstimator

onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
#onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])

flights = onehot.fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['km'], outputCol='features')
flights = assembler.transform(flights)
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

flights.show(5, truncate=False)
print("Subset from the flights DataFrame:")
flights.select('km', 'features', 'duration').show(5, truncate=False)

# Code added by Amit
print(flights_train.printSchema())


+---+---+---+-------+------+---+------+--------+-----+------+-------+-------------+--------+
|mon|dom|dow|carrier|flight|org|depart|duration|delay|km    |org_idx|org_dummy    |features|
+---+---+---+-------+------+---+------+--------+-----+------+-------+-------------+--------+
|11 |20 |6  |US     |19    |JFK|9.48  |351     |null |3465.0|2.0    |(7,[2],[1.0])|[3465.0]|
|0  |22 |2  |UA     |1107  |ORD|16.33 |82      |30   |509.0 |0.0    |(7,[0],[1.0])|[509.0] |
|2  |20 |4  |UA     |226   |SFO|6.17  |82      |-8   |542.0 |1.0    |(7,[1],[1.0])|[542.0] |
|9  |13 |1  |AA     |419   |ORD|10.33 |195     |-5   |1989.0|0.0    |(7,[0],[1.0])|[1989.0]|
|4  |2  |5  |AA     |325   |ORD|8.92  |65      |null |415.0 |0.0    |(7,[0],[1.0])|[415.0] |
+---+---+---+-------+------+---+------+--------+-----+------+-------+-------------+--------+
only showing top 5 rows

Subset from the flights DataFrame:
+------+--------+--------+
|km    |features|duration|
+------+--------+--------+
|3465.0|[3465.0]|351  

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Create a regression object and train on training data
regression = ____(____).____(____)

# Create predictions for the testing data and take a look at the predictions
predictions = ____.____(____)
predictions.select('duration', 'prediction').show(5, False)

# Calculate the RMSE
____(____).____(predictions)

In [4]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Create a regression object and train on training data
regression = LinearRegression(labelCol = 'duration').fit(flights_train)

# Create predictions for the testing data and take a look at the predictions
predictions = regression.transform(flights_test)
predictions.select('duration', 'prediction').show(5, False)

# Code added by Amit
print(predictions.printSchema())

# Calculate the RMSE
RegressionEvaluator(labelCol = 'duration').evaluate(predictions)

+--------+------------------+
|duration|prediction        |
+--------+------------------+
|385     |359.2883184020809 |
|135     |149.918520767347  |
|200     |224.07190048619026|
|64      |72.96547263054742 |
|259     |269.16926202948673|
+--------+------------------+
only showing top 5 rows

root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- org_idx: double (nullable = false)
 |-- org_dummy: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)

None


17.038497996029204

You've built a simple regression model. Let's make sense of the coefficients!